# Lesson 1: Slopes. The derivative (Newton & Leibniz, late 1600s)

## The old idea

In the 1600s, Isaac Newton and Gottfried Leibniz, independently, solved a problem that had frustrated mathematicians since the ancient Greeks: **how fast is something changing *right now*?** A falling stone doesn't have one speed. It speeds up every instant. "Speed at this exact moment" sounds meaningless (a moment has no duration to measure distance over), yet physics needed it. Their answer: measure the speed over a shorter and shorter stretch of time, and see what number the answers close in on. That number is the **derivative**.

## What it becomes in machine learning

Training a model means adjusting numbers (weights) to make an error smaller. The question "if I nudge this weight up a tiny bit, does the error go up or down, and by how much?" is *exactly* "how fast is the error changing as this weight changes?". The answer is a derivative. Every learning algorithm in this series (linear regression, neural networks, all of it) is built on it. In ML it's usually called a **gradient** (Lesson 2 shows the gradient is just derivatives collected into a list).

## How to use this notebook

Whenever you see a code block, there's an empty code cell right under it. **Type the code yourself** (don't copy-paste), run it, and compare with the expected output shown after. Read the explanations before the code, not after. Variable names are long on purpose so the code reads like the maths.

**By the end of this lesson you can:** compute a slope numerically by "wiggling", derive a slope exactly by hand for powers of *x*, read the notation `f'(x)`, `dy/dx`, and `f''(x)`, and explain why slope tells you which way is downhill.

---
## Step 1: Average speed over a stretch of time

Galileo found (around 1600) that a dropped stone falls a distance of about `4.9 * t**2` metres after `t` seconds. (The 4.9 is half of gravity's 9.8 m/s².) Type this function:

```python
def distance_fallen(seconds):
    return 4.9 * seconds ** 2

print(distance_fallen(1))
print(distance_fallen(3))
```

In [ ]:
# type it here

**Expected output:**
```text
4.9
44.1
```

After 1 second it has fallen 4.9 m; after 3 seconds, 44.1 m. **Average speed** over a stretch is distance covered divided by time taken. Here: `(44.1 - 4.9) / (3 - 1) = 19.6` m/s. Write that as a reusable function. It takes *any* function as input (functions are values in Python, so you can pass them around):

```python
def average_speed(function, start_time, end_time):
    distance_covered = function(end_time) - function(start_time)
    time_taken = end_time - start_time
    return distance_covered / time_taken

print(average_speed(distance_fallen, 1, 3))
```

In [ ]:
# type it here

**Expected output:**
```text
19.6
```

**Signature explained:** `average_speed` expects three things: a function (anything that takes one number and returns one number), and two numbers. Inside, `function(end_time)` just *calls* whatever function you handed in.

This number is the **slope of a straight line** drawn between the two points on the graph of distance against time ("rise over run": rise = distance covered, run = time taken).

**Try this (new cell):** call `average_speed(distance_fallen, 0, 1)`, then `(1, 2)`, then `(2, 3)`. Predict first: should the numbers go up or down? Why?

---
## Step 2: Shrink the stretch and watch what happens

We want the speed at exactly t = 1 second. Take the stretch from 1 to 1 + gap and make the gap smaller and smaller. Type:

```python
for gap in [1, 0.1, 0.01, 0.001, 0.0001, 0.00001]:
    speed = average_speed(distance_fallen, 1, 1 + gap)
    print(gap, speed)
```

In [ ]:
# type it here

**Expected output:**
```text
1 14.700000000000001
0.1 10.29
0.01 9.848999999999988
0.001 9.804899999999975
0.0001 9.800489999999243
1e-05 9.800048999977408
```

The speeds close in on **9.8**. That's the speed at exactly t = 1: the stretch got shorter and shorter, and the answer stopped changing much. "The number the answers close in on" has a name: a **limit**.

**A trap to know about:** computers store numbers with limited precision. Shrinking the gap forever doesn't keep improving the answer. Try gaps far smaller than you'd normally use:

```python
for gap in [1e-8, 1e-10, 1e-12, 1e-14, 1e-15]:
    speed = average_speed(distance_fallen, 1, 1 + gap)
    print(gap, speed)
```

In [ ]:
# type it here

**Expected output:**
```text
1e-08 9.799999982236432
1e-10 9.8
1e-12 9.800177619893429
1e-14 9.777777777777779
1e-15 9.6
```

(`1e-8` means 0.00000001.) The answers get *worse*, then wrong. When `gap` is tiny, `1 + gap` and `1` are almost the same number, and subtracting two nearly equal floating-point numbers destroys the digits that matter. Rule of thumb for this series: a gap around `0.0001` to `0.000001` is a good balance. That's why you'll see `0.0001` everywhere in ML code that checks slopes numerically.

---
## Step 3: The definition, in symbols and in code

What you just did is the definition of the derivative. In a textbook:

```
f'(x) = limit as h→0 of  [ f(x + h) - f(x) ] / h
```

Read it piece by piece, with the code you already wrote:

| Symbol | Meaning | Your code |
|---|---|---|
| `f` | the function | `function` |
| `x` | the point where you want the slope | `1` (start_time) |
| `h` | the tiny gap | `gap` |
| `f(x+h) - f(x)` | rise | `distance_covered` |
| `[ ... ] / h` | rise over run | `distance_covered / time_taken` |
| `limit h→0` | make `h` smaller and smaller, see where it settles | the shrinking loop |

`f'(x)` ("f prime of x") is Newton-style shorthand. Leibniz wrote the same thing as `df/dx` ("a tiny change in f over a tiny change in x"). Both mean: **the slope of the function at the point x.** Write a function that does the whole thing, with a small fixed gap standing in for the limit:

```python
def slope_by_wiggling(function, point, gap=0.0001):
    return (function(point + gap) - function(point)) / gap

print(slope_by_wiggling(distance_fallen, 1))
print(slope_by_wiggling(distance_fallen, 2))
print(slope_by_wiggling(distance_fallen, 3))
```

In [ ]:
# type it here

**Expected output:**
```text
9.800489999998163
19.6004900000446
29.40049000009992
```

`gap=0.0001` in the signature is a **default value**: if you don't pass a gap, it uses 0.0001. The speeds at 1, 2, and 3 seconds are about 9.8, 19.6, and 29.4: the stone speeds up by 9.8 m/s every second. This one function ("wiggle the input a tiny bit, see how the output moves") works on *any* function. Keep it, we'll reuse it all series.

---
## Step 4: The exact answer, by hand (no computer needed)

Wiggling is approximate. Newton and Leibniz found you can get the exact answer with algebra. Take the simplest interesting function, `f(x) = x²`. Plug it into the definition:

```
[ f(x + h) - f(x) ] / h
= [ (x + h)² - x² ] / h
= [ x² + 2xh + h² - x² ] / h        (expand (x+h)²)
= [ 2xh + h² ] / h                  (the x² cancels)
= 2x + h                            (divide everything by h)
```

Now let `h` shrink to 0: the leftover `h` vanishes and **the slope of x² is exactly 2x**. So the stone's speed after `t` seconds is `4.9 * 2t = 9.8t`. At t = 1 that's 9.8. ✓ Matches what the numbers closed in on.

The same trick works for other powers. The pattern (the **power rule**) is:

```
slope of x^n  is  n * x^(n-1)
```

Check: n=2 gives `2 * x^1 = 2x` ✓. And two more rules that need no new idea: the slope of `3 * (something)` is 3 times the slope of the something, and the slope of a sum is the sum of the slopes. A constant (like +7) doesn't change as x changes, so its slope is 0.

Try them on `f(x) = 3x³ - 2x + 7`:

```
slope = 3 * (3x²) - 2 * (1) + 0 = 9x² - 2
```

Now let the computer check your algebra against the wiggle:

```python
def curve(x):
    return 3 * x ** 3 - 2 * x + 7

def curve_slope_exact(x):
    return 9 * x ** 2 - 2

for point in [0, 1, 2, -1.5]:
    print(point, slope_by_wiggling(curve, point), curve_slope_exact(point))
```

In [ ]:
# type it here

**Expected output:**
```text
0 -1.9999999700015536 -2
1 7.000900029989765 7
2 34.001800030054596 34
-1.5 18.24865003001719 18.25
```

The two columns agree to several decimal places at every point. When your hand-derived formula matches the wiggle, you've derived it correctly. This **"derive by hand, verify by wiggling"** habit will carry you through every derivation in this series, including the big ones in neural networks.

**Try this (new cell):** the power rule says the slope of `x**0.5` (square root) is `0.5 * x**(-0.5)`. Check it numerically at x=4 and x=9. (Hint: `x ** 0.5` and `x ** -0.5` work in Python.) Does the rule work for non-whole-number powers?

---
## Step 5: What the sign of the slope tells you

Here's the idea machine learning leans on. Take the bowl `f(x) = (x - 3)²`, whose lowest point is at x = 3. Its slope is `2 * (x - 3)`. You can get that with only the Step 4 rules: expand `(x - 3)²` into `x² - 6x + 9`, then the slope is `2x - 6`, which is `2 * (x - 3)`. (Lesson 2 gives a faster route, the chain rule.) Type:

```python
def bowl(x):
    return (x - 3) ** 2

def bowl_slope(x):
    return 2 * (x - 3)

for point in [0, 2, 3, 4, 7]:
    print(point, bowl(point), bowl_slope(point))
```

In [ ]:
# type it here

**Expected output:**
```text
0 9 -6
2 1 -2
3 0 0
4 1 2
7 16 8
```

Read the table:

- At x = 0 the slope is **-6** (negative): the curve is going *down* as x increases, so the bottom is to the **right**.
- At x = 7 the slope is **+8** (positive): the curve is going *up*, so the bottom is to the **left**.
- At x = 3 the slope is **0**: flat. That's the bottom.
- The bigger the slope's size, the steeper the hill.

So the slope is a **compass**: its sign says which way is uphill, its size says how steep. To go *down* the bowl, move **against** the slope. A tiny preview of where this goes (Lesson 4 does it properly):

```python
position = 0.0
step_size = 0.1

for step_number in range(1, 8):
    position = position - step_size * bowl_slope(position)
    print(step_number, round(position, 4))
```

In [ ]:
# type it here

**Expected output:**
```text
1 0.6
2 1.08
3 1.464
4 1.7712
5 2.017
6 2.2136
7 2.3709
```

It walks toward 3. If you did Lesson 0 (the `nudge` function), this might look familiar: `position - step_size * 2 * (position - 3)` is `nudge(position, 3, 2 * step_size)`. Walking downhill using the slope *is* a nudge toward the bottom. Lesson 4 explains why that works and when it breaks.

---
## Step 6: The slope of the slope (curvature)

You can take a slope of a slope. For the bowl, the slope is `2(x - 3)`, and *its* slope is 2 everywhere. That's the **second derivative**, written `f''(x)`. It measures **curvature**: how quickly the slope itself changes.

- A gentle bowl like `(x-3)²` has `f'' = 2`.
- A narrow, steep bowl like `10 * (x-3)²` has `f'' = 20`.

Curvature is what limits how big a step you can safely take downhill (Lesson 4). Compute it numerically by wiggling the slope function:

```python
def second_slope_by_wiggling(function, point, gap=0.001):
    slope_a_bit_right = slope_by_wiggling(function, point + gap)
    slope_a_bit_left = slope_by_wiggling(function, point - gap)
    return (slope_a_bit_right - slope_a_bit_left) / (2 * gap)

print(second_slope_by_wiggling(bowl, 0))
print(second_slope_by_wiggling(bowl, 5))
print(second_slope_by_wiggling(curve, 2))
```

In [ ]:
# type it here

**Expected output:**
```text
1.999999987845058
1.9999999989472883
36.00090003885725
```

The bowl gives about 2 at both points: curvature is constant for a parabola. For `curve` (the 3x³ - 2x + 7 from Step 4) the slope formula was `9x² - 2`, so the second derivative is `18x`, which is 36 at x = 2. Compare with the number printed on the last line.

---
## Step 7: Guided exploration: other functions

So far we've only used powers. Slopes work on any smooth function. Copy your Step 4 check into a new cell and swap in these named alternatives. For each, predict the slope before running.

1. `import math`, then try `math.exp` (e to the power x). Print `slope_by_wiggling(math.exp, point)` and `math.exp(point)` for points 0, 1, 2. What do you notice? (This function is its own slope, which is why it's everywhere in probability and in neural networks.)
2. Try `math.sin` and compare its wiggled slope against `math.cos`.
3. Try `abs` (absolute value, a V shape) at the points -2, 2 and 0. What slope does it report at 0, and why is that a problem for "walk against the slope"? (This returns in Lesson 2 when we meet an alternative to squared error.)

```python
import math

for point in [0, 1, 2]:
    print(point, slope_by_wiggling(math.exp, point), math.exp(point))
```

In [ ]:
# type it here

**Expected output:**
```text
0 1.000050001667141 1.0
1 2.718417747082924 2.718281828459045
2 7.389425564063856 7.38905609893065
```

---
## Step 8: Challenge

A ball is thrown straight up, and its height after `t` seconds is `height(t) = 20 * t - 4.9 * t**2` metres.

1. Using the rules from Step 4, work out the slope formula `height_slope(t)` **on paper**.
2. The ball is at its highest point when its upward speed is zero, meaning when the slope is zero. Solve for that `t` on paper.
3. In a new cell, write `height`, then use `slope_by_wiggling` to check your answer from (2): the slope there should be about 0, and the height there should be larger than just before or after.
4. What does the slope say at t = 0 (the throw), and at t = 3? What do the signs mean physically?

Write answers on paper first, then type the code. Solutions are in the last cell below.

```python
def height(seconds):
    return 20 * seconds - 4.9 * seconds ** 2

# your attempt below: height_slope, then the time of the peak, then checks
```

In [ ]:
# type it here

---
## Where this goes next

You now have one tool: **a way to measure how fast anything changes**, plus the rule that slope is a compass pointing uphill. Lesson 2 extends it to functions with several inputs (a model with many weights) and to functions *of* functions (a model that feeds one calculation into the next). That is the **chain rule**, and it's the entire mathematical content of backpropagation.

Once Steps 1-6 feel solid, one thing to probe on your own: in Step 2 the "wrong" answers at tiny gaps came from subtracting nearly equal numbers. Search "catastrophic cancellation" and "central difference" to find out how professionals get more accurate slopes from the same idea. (Try replacing `slope_by_wiggling` with `(f(x+gap) - f(x-gap)) / (2*gap)` on your `distance_fallen` and see whether it's more accurate.)

---
# Solutions to the Step 8 challenge (try it first)

1. `height_slope(t) = 20 - 9.8 * t`. (Slope of `20t` is 20; slope of `4.9 t²` is `4.9 * 2t = 9.8t`.)
2. Slope = 0 when `20 - 9.8t = 0`, so `t = 20 / 9.8 ≈ 2.0408` seconds.
3. Check:

```python
def height_slope(t):
    return 20 - 9.8 * t

peak_time = 20 / 9.8
print(peak_time)
print(slope_by_wiggling(height, peak_time))
print(height(peak_time - 0.1), height(peak_time), height(peak_time + 0.1))
print(height_slope(0), height_slope(3))
```

**Output:**
```text
2.0408163265306123
-0.0004899999339613714
20.35916326530612 20.40816326530612 20.35916326530612
20.0 -9.400000000000002
```

The wiggled slope at the peak prints about -0.0005, not exactly 0. That's the forward-wiggle's small bias (it looks slightly ahead, where the ball has begun to fall), not a mistake in your answer.
4. At t = 0 the slope is +20: the ball moves upward at 20 m/s (it was just thrown). At t = 3 the slope is about -9.4: moving *downward* at 9.4 m/s (it has already peaked and is falling). The sign flips from + to - exactly at the peak, which is why "slope equals zero" finds tops and bottoms. In ML we look for the **bottom** of an error curve, and this is how.